# 01 · Download SoccerNet 224p video for the 25-game baseline set
**One-time job. Run with GPU OFF (this step needs no GPU — saves quota).**

Produces a persistent private dataset `soccernet-25games` containing all 25 games' `1_224p.mkv` / `2_224p.mkv` + `Labels-v2.json`, so notebooks 02/03 reuse it.

**Kaggle settings before running:**
- Accelerator: **None** (no GPU)
- Internet: **ON**
- Add data: attach **`daniels02/3-matches`** (so the 3 already-downloaded games are reused, not re-fetched)
- Add-ons → Secrets: enable **`SOCCERNET_PWD`**

## Guard cell — fail fast if the session is misconfigured

In [ ]:
import os
from pathlib import Path

# 1. NDA password secret present?
try:
    from kaggle_secrets import UserSecretsClient
    assert UserSecretsClient().get_secret("SOCCERNET_PWD"), "SOCCERNET_PWD secret is empty"
    print("OK  SOCCERNET_PWD secret present")
except Exception as e:
    raise SystemExit("FAIL secret check: " + repr(e) + "  -> Add-ons -> Secrets -> add SOCCERNET_PWD")

# 2. Internet on? (needed to git clone + download)
import urllib.request
try:
    urllib.request.urlopen("https://github.com", timeout=10)
    print("OK  internet reachable")
except Exception as e:
    raise SystemExit("FAIL internet: " + repr(e) + "  -> enable Internet in notebook settings")

# 3. 3-matches dataset attached? (optional but recommended)
hits = list(Path("/kaggle/input").rglob("*_224p.mkv")) if Path("/kaggle/input").exists() else []
msg = (f"OK  found {len(hits)} existing .mkv under /kaggle/input (3-matches expected = 6)"
       if hits else "WARN no existing videos attached - all 25 will be downloaded fresh")
print(msg)


## Clone repo + install deps (thin runner — code lives in GitHub)

In [ ]:
import os
REPO = "https://github.com/DanielSch02/Football_JEPA.git"
%cd /kaggle/working
!rm -rf Football_JEPA
!git clone {REPO}
%cd Football_JEPA
!git log --oneline -1
!pip install -q SoccerNet


## Seed working dir with the 3 already-downloaded games
Copy the attached `3-matches` videos into the working dir so the final dataset is complete (all 25). `load_videos.py` will then skip them (already present) and only fetch the other 22.

In [ ]:
import shutil
from pathlib import Path
from footy.config import VJEPA_GAMES_FULL, default_data_dir

WORK = Path("/kaggle/working/soccernet")
# default_data_dir() auto-detects the attached 3-matches mount
src_root = Path(default_data_dir())
print("source (existing videos):", src_root)

copied = 0
for game, _ in VJEPA_GAMES_FULL:
    for fname in ("Labels-v2.json", "1_224p.mkv", "2_224p.mkv"):
        s = src_root / game / fname
        d = WORK / game / fname
        if s.exists() and not d.exists():
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(s, d); copied += 1
print(f"seeded {copied} files from existing dataset")


## Download the remaining 22 games (224p video + labels)
Resumable: files already present (the seeded 3 games) are skipped. ~13 GB total, takes a while.

In [ ]:
!python -m scripts.load_videos --full --data_dir /kaggle/working/soccernet

## Verify all 25 games are complete

In [ ]:
import glob
from footy.config import VJEPA_GAMES_FULL
WORK = "/kaggle/working/soccernet"
ok = 0
for game, _ in VJEPA_GAMES_FULL:
    have = [f for f in ("1_224p.mkv", "2_224p.mkv", "Labels-v2.json")
            if glob.glob(f"{WORK}/{game}/{f}")]
    if len(have) == 3:
        ok += 1
    else:
        print("MISSING", game, have)
print()
print(ok, "/25 games complete")
n_mkv = len(glob.glob(f"{WORK}/**/*_224p.mkv", recursive=True))
print("total mkv files:", n_mkv, "(expect 50)")


## Save as a private Kaggle dataset `soccernet-25games`
Uses the API method (reliable). Run once to create; re-run uses `version` to update.

In [ ]:
import json, os
WORK = "/kaggle/working/soccernet"
meta = {"title": "soccernet-25games",
        "id": "daniels02/soccernet-25games",
        "licenses": [{"name": "CC0-1.0"}]}
json.dump(meta, open(f"{WORK}/dataset-metadata.json","w"))

# First time: create. If it already exists, use the 'version' line instead.
!kaggle datasets create -p {WORK} --dir-mode tar
# !kaggle datasets version -p {WORK} -m "all 25 games" --dir-mode tar
